# Effect of Top Actors on Movie Success: A Regression Analysis

### Research Question: What is the estimated effect of having top actors on a movie’s probability of success?

#### Dataset: TMDB 5000 Movie Dataset (Kaggle)
#### Definition of Success: A movie is considered successful if its revenue is in the top 25% of all movies in the dataset.

In [2]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import statsmodels.api as sm
from collections import Counter
import ast

df = pd.read_csv('../data/movies.csv')

Defining Success (top 25% revenue)

In [3]:
threshold = df["revenue"].quantile(0.75)
df["success"] = (df["revenue"] >= threshold).astype(int)

Release Date Parsing

In [4]:
df["release_date"] = pd.to_datetime(df["release_date"])
df["release_month"] = df["release_date"].dt.month
df["release_year"] = df["release_date"].dt.year

Actor Counts

In [5]:
df["cast_parsed"] = df["cast"].apply(ast.literal_eval)
df["actor_names"] = df["cast_parsed"].apply(lambda x: [d["name"] for d in x])

all_actors = [actor for sublist in df["actor_names"] for actor in sublist]

actor_counts = Counter(all_actors)
common_20_actors = actor_counts.most_common(20)
top_20_actors = [name[0] for name in actor_counts.most_common(20)]
df["num_top_actors"] = df["actor_names"].apply(
    lambda actors: sum(1 for actor in actors if actor in top_20_actors)
)

df['has_top_actor'] = (df['num_top_actors'] > 0).astype(int)

# Note: For this project, I use high frequency in movies as a proxy for whether an actor is a "top" actor; in the real world, this is a simplifying assumption.

Genre Names

In [6]:
# One-Hot Encoding the Genre Names

df["genres_parsed"] = df["genres"].apply(ast.literal_eval)
df["genre_names"] = df["genres_parsed"].apply(lambda x: [d["name"] for d in x])

mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df["genre_names"])
genre_df = pd.DataFrame(genres_encoded, columns=mlb.classes_, index=df.index)
df = pd.concat([df, genre_df], axis=1).dropna()

#### Variables <br>
Treatment: movie has top actor<br><br>
Outcome: movie success<br><br>
Confounders: budget, genre, release timing<br><br>

In [7]:
covariates = ['budget', 'release_year', 'release_month'] + list(mlb.classes_)
X = df[covariates]
X = sm.add_constant(X)
y = df['success']

X['treatment'] = df['has_top_actor']

model = sm.Logit(y, X)
result = model.fit()
print(result.summary())

margins = result.get_margeff()
print(margins.summary())

         Current function value: 0.399816
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 1494
Model:                          Logit   Df Residuals:                     1469
Method:                           MLE   Df Model:                           24
Date:                Sun, 17 May 2026   Pseudo R-squ.:                  0.4106
Time:                        20:30:26   Log-Likelihood:                -597.33
converged:                      False   LL-Null:                       -1013.5
Covariance Type:            nonrobust   LLR p-value:                2.963e-160
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              84.3479     16.074      5.247      0.000      52.843     115.853
budget           5.165e-08   3.34e-09     15.454  

/Users/raimaazrafiislam/opt/miniconda3/envs/newenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Interpreting Logit Marginal Effects

The following summarizes the average marginal effects from the logistic regression. Marginal effects approximate the change in probability of success (top 25% revenue) for a one-unit increase in each variable.

### 1. Treatment: Top Actors
- **Coefficient:** 0.0503  
- **p-value:** 0.041

Movies with a top actor have about a **5.0% higher probability of being successful**, holding budget, release timing, and genres constant.

---

### 2. Budget
- **Coefficient:** 6.595e-09
- **p-value:** 0.000 

Higher-budget movies are more likely to be successful. The effect is small per unit of budget ($1), but meaningful at larger scales.

**At the million-dollar scale:** 
6.595e-09 × 1,000,000 ≈ 0.0066

- A **$1 million increase in budget** is associated with an approximate **0.66% increase in the probability** of being in the top 25% of revenue, holding other factors constant.

---

### 3. Release Timing

#### Release Year
- **Coefficient:** -0.0055 
- **p-value:** 0.000 

Movies released in later years are slightly less likely (about 0.55%) to reach the top 25% of revenue.

#### Release Month:
- **Coefficient:** 0.0055
- **p-value:** 0.052 

The month that a movie is released does not likely have an impact on the movie's chances of success (p > 0.05).

---

### 4. Genre

The only genres that were statistically significant (p < 0.05) were Crime, Drama, Science Fiction, Thriller, and Western.

#### Crime
- **Coefficient:** -0.0776
- **p-value:** 0.009

- Crime movies have about 7.8% lower probability of being top-revenue films compared with the baseline genre.

#### Drama
- **Coefficient:** -0.0570
- **p-value:** 0.019

- Drama movies have about 5.7% lower probability of being top-revenue films compared with the baseline genre.

#### Science Fiction
- **Coefficient:** -0.0787 
- **p-value:** 0.009

- Science Fiction movies have about 7.9% lower probability of being top-revenue films compared with the baseline genre.

#### Thriller
- **Coefficient:** -0.0532 
- **p-value:** 0.034

- Thriller movies have about 5.3% lower probability of being top-revenue films compared with the baseline genre.

#### Western
- **Coefficient:** -0.2643
- **p-value:** 0.016 

- Western have about 26.4% lower probability of being top-revenue films compared with the baseline genre.

## Summary

#### Original Question: 
Does casting a top actor increase the probability that a movie becomes a financial success?

#### Answer: 
The logistic regression suggests that having a top actor has a small but statistically significant positive effect on movie success. After controlling for budget, release timing, and genre, movies with top actors have approximately 5% higher probability of being in the top revenue quartile. However, budget is a  stronger predictor of success. Additionally, the model did not fully converge -- results should be interpreted with caution.